In [ ]:
"""
QPE Pure-NumPy Reference Simulator
====================================
Runs phase estimation via exact statevector simulation.
No SpinQ or Qiskit required — useful for quick testing and
generating the comparison data displayed in the interactive dashboard.

Usage:
    python qpe_simulator.py
"""

import math
import numpy as np
from collections import Counter


def qft_matrix(n: int) -> np.ndarray:
    """Return the n-qubit QFT unitary as a 2^n × 2^n matrix."""
    N = 2 ** n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega ** (j * k) for k in range(N)] for j in range(N)]) / np.sqrt(N)
    return F


def simulate_qpe(theta: float, n_count: int = 4, shots: int = 4096) -> dict:
    """
    Exact statevector QPE simulation.

    Args:
        theta:   Rz rotation angle (radians)
        n_count: counting qubits
        shots:   measurement samples

    Returns dict with keys: true_phase, estimated_phase, error,
                            probabilities (array len 2^n_count), counts
    """
    N = 2 ** n_count
    true_phase = (theta / (2 * math.pi)) % 1.0

    # The counting register starts as uniform superposition |+⟩^⊗n
    # After controlled-Rz^{2^k} and inverse QFT, the ideal state is
    # a peaked distribution centred on the integer closest to φ·2^n.
    #
    # Exact amplitude at output bitstring |m⟩:
    #
    #   A(m) = (1/N) Σ_{k=0}^{N-1} exp(2πi k (φ - m/N))
    #
    # This is the Dirichlet kernel evaluated at φ − m/N.

    phi_int = true_phase * N        # ideal non-integer index
    m_vals  = np.arange(N)

    def amplitude(m):
        delta = phi_int - m
        if abs(delta % N) < 1e-9:   # exactly representable
            return 1.0 + 0j
        # Dirichlet kernel (wraps mod N)
        return (1 / N) * (1 - np.exp(2j * np.pi * delta)) / (1 - np.exp(2j * np.pi * delta / N))

    amps  = np.array([amplitude(m) for m in m_vals])
    probs = np.abs(amps) ** 2
    probs /= probs.sum()            # numerical normalisation

    # Sample
    samples = np.random.choice(N, size=shots, p=probs)
    counts_raw = Counter(int(s) for s in samples)

    best_m = int(probs.argmax())
    est_phase = best_m / N

    return {
        "true_phase": true_phase,
        "estimated_phase": est_phase,
        "error": abs(est_phase - true_phase),
        "probabilities": probs.tolist(),
        "counts": {format(k, f'0{n_count}b'): v for k, v in counts_raw.items()},
        "n_count": n_count,
        "resolution": 1 / N,
    }


def run_comparison_table(angles, n_count=4, shots=4096):
    """Print a comparison table for a list of rotation angles."""
    N = 2 ** n_count
    res = 1 / N
    header = f"{'θ (rad)':>12}  {'True φ':>8}  {'Est. φ':>8}  {'Error':>8}  {'Exact?':>7}"
    print(f"\n{'QFT Phase Estimation — NumPy Simulator':^60}")
    print(f"n_count={n_count}  →  resolution = 1/2^{n_count} = {res:.4f}")
    print("─" * 60)
    print(header)
    print("─" * 60)
    for theta in angles:
        r = simulate_qpe(theta, n_count, shots)
        exact = "✓" if r["error"] < res / 2 else "✗ aliased"
        print(f"{theta:>12.5f}  {r['true_phase']:>8.5f}  {r['estimated_phase']:>8.5f}"
              f"  {r['error']:>8.5f}  {exact:>9}")
    print("─" * 60)


if __name__ == "__main__":
    angles = [
        math.pi / 4,          # φ = 0.125  — 4-bit exact
        math.pi / 3,          # φ ≈ 0.1667 — aliased
        math.pi,              # φ = 0.5    — 4-bit exact
        3 * math.pi / 2,      # φ = 0.75   — 4-bit exact
        1.2,                  # generic
        0.7,
        5 * math.pi / 7,      # irrational fraction
    ]
    run_comparison_table(angles, n_count=4)
    run_comparison_table(angles, n_count=6)


           QFT Phase Estimation — NumPy Simulator           
n_count=4  →  resolution = 1/2^4 = 0.0625
────────────────────────────────────────────────────────────
     θ (rad)    True φ    Est. φ     Error   Exact?
────────────────────────────────────────────────────────────
     0.78540   0.12500   0.12500   0.00000          ✓
     1.04720   0.16667   0.18750   0.02083          ✓
     3.14159   0.50000   0.50000   0.00000          ✓
     4.71239   0.75000   0.75000   0.00000          ✓
     1.20000   0.19099   0.18750   0.00349          ✓
     0.70000   0.11141   0.12500   0.01359          ✓
     2.24399   0.35714   0.37500   0.01786          ✓
────────────────────────────────────────────────────────────

           QFT Phase Estimation — NumPy Simulator           
n_count=6  →  resolution = 1/2^6 = 0.0156
────────────────────────────────────────────────────────────
     θ (rad)    True φ    Est. φ     Error   Exact?
────────────────────────────────────────────────────────────
     